Example trapped resonance objective function computation and optimization script

In [ ]:
# Objective function computation

import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
from desc.examples import get


# eq = desc.io.load("equil_Helios_E0956_R80_B60_DESC_fixed.h5")
eq = get('HELIOTRON')

obj = TrappedResonance(
    eq, 
    num_eta=2, 
    M=1, 
    N=0,
    num_transit=8,
    knots_per_transit=100,
    num_quad=48,
    weight_method="bump",
    cropping_DOmega=True,
)

obj.build()
value = obj.compute(eq.params_dict)
print(value)

In [ ]:
# Example optimization

import sys
import os
import math

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    ForceBalance,
    QuasisymmetryTripleProduct,
    GammaC,
    TrappedResonance,
    FixCurrent
)
from desc.optimize import Optimizer
from desc.examples import get



# load initial equilibrium
# eq_init = desc.io.load("desc_eq_beta2.5_QA.h5") # QA
eq_init = get('ESTELL')
eq_init.change_resolution(L=4,M=4,N=4,L_grid=8,M_grid=8,N_grid=8) # change resolution to make optimization faster, grid resolution needs to change explicitly
eq_init.surface = eq_init.get_surface_at(1.0)

# Specify equilibrium nfp and helicity
N = 0 # QA
nfp = eq_init.NFP

# optimizer = Optimizer("proximal-scipy-bfgs")
optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)

eq_qs_T = eq_init.copy()  # make a copy of the original equilibria

constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixCurrent(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# Create grid for QSTripleProduct objective 
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)

# Objective for resonance
objective_fT = ObjectiveFunction(
    (
        QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol),
        # TrappedResonance(
        #     eq_qs_T,
        #     num_rho=10,
        #     num_eta=5,
        #     KE_frac=1,
        #     num_transit=4,
        #     num_quad=32,
        #     num_pitch=4,
        #     num_well=None,
        #     pitch_invs=None,
        #     N=0,
        #     M=1,
        #     p_max=10,
        #     q_max=10,
        #     res_range_min=-4,
        #     res_range_max=4,
        #     DEBUG=False,
        #     f_q_conservative=False,
        #     weight_method="bump",
        #     Delta_Omega=None,
        #     fill_value=11,
        #     wd_blur=1.25,
        #     stab_sacrifice=True,
        #     jac_chunk_size = 1
        # )
        TrappedResonance(
            eq=eq_qs_T, 
            num_eta=2, 
            M=1, 
            N=0,
            num_well=2, 
            num_transit=8,
            knots_per_transit=100,
            num_quad=48,
        )
    ),
)

eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=10,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

In [8]:
eq0 = eq_init.copy()      # or the actual pre-opt eq used in optimize
eqf = eq_qs_T             # optimized eq object returned/updated by optimize
x0 = objective_fT.x(eq0)  # full-state x in objective_fT coordinates
xf = objective_fT.x(eqf)
J0 = objective_fT.jac_unscaled(x0, objective_fT.constants)
Jf = objective_fT.jac_unscaled(xf, objective_fT.constants)
print("x0 shape:", x0.shape, "J0:", J0.shape)
print("xf shape:", xf.shape, "Jf:", Jf.shape)

x0 shape: (306,) J0: (435, 306)
xf shape: (306,) Jf: (435, 306)


In [11]:
print("J0:", J0)
print("Jf:", Jf)

J0: [[-8.18929586e-02 -1.83324959e-02 -8.32042430e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.28370392e-01  1.34413929e-01  1.06089793e-01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-1.75990898e-01 -4.54659588e-01 -3.42439554e-02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 3.05522722e+01 -1.57144033e+02  6.62347370e+01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 2.04683613e+00 -1.58402465e+01  2.77476718e+01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.68643393e+00  9.59934852e+00 -2.87513094e+01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]
Jf: [[-8.18929586e-02 -1.83324959e-02 -8.32042430e-04 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-2.28370392e-01  1.34413929e-01  1.06089793e-01 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [-1.75990898e-01 -4.54659588e-01 -3.42439554e-02 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 3.05522722e+01 -1.57144033e+02  6